# Aula 02: pipeline de experimento A/B

Este notebook acompanha a aula de causalidade. O objetivo nao e fazer inferencia estatistica formal ainda. A ideia e entender, com tabelas e graficos, como uma pergunta causal vira uma comparacao entre tratamento e controle.

## 1. Pergunta causal

**Enviar um lembrete dois dias antes do prazo aumenta a chance de estudantes entregarem uma lista no prazo?**

- Unidade observacional: estudante.
- Tratamento: receber lembrete.
- Controle: nao receber lembrete extra.
- Resultado: entregar a lista no prazo.
- Comparacao: taxa de entrega no grupo tratado contra taxa de entrega no grupo controle.

## 2. Por que precisamos de controle?

Se apenas observarmos quem recebeu lembrete e quem entregou, podemos confundir o efeito do lembrete com outras diferencas entre estudantes. Por exemplo, estudantes mais organizados poderiam abrir mais mensagens e tambem entregar mais no prazo.

A randomizacao ajuda a criar dois grupos comparaveis antes do tratamento.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 160

dados = pd.DataFrame({
    "estudante": np.arange(1, n + 1),
    "periodo": rng.choice([1, 2, 3, 4, 5], size=n, p=[0.25, 0.25, 0.20, 0.18, 0.12]),
    "listas_anteriores_no_prazo": rng.binomial(4, 0.62, size=n),
    "creditos_semestre": rng.choice([12, 16, 20, 24, 28], size=n, p=[0.10, 0.25, 0.35, 0.20, 0.10])
})

dados["grupo"] = rng.choice(["controle", "lembrete"], size=n)
dados.head()

## 3. Os grupos parecem comparaveis?

Randomizacao nao cria grupos identicos. Ela cria grupos sem uma regra enviesada de escolha. Mesmo assim, vale olhar se as covariaveis observadas parecem parecidas.

In [ ]:
resumo_grupos = dados.groupby("grupo")[[
    "periodo",
    "listas_anteriores_no_prazo",
    "creditos_semestre"
]].mean()

resumo_grupos.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

for ax, coluna, titulo in zip(
    axes,
    ["periodo", "listas_anteriores_no_prazo", "creditos_semestre"],
    ["Periodo medio", "Listas anteriores", "Creditos no semestre"]
):
    resumo_grupos[coluna].plot(kind="bar", ax=ax, color=["#4c78a8", "#f58518"])
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()

## 4. Observando o resultado

Agora observamos quem entregou no prazo. Em um experimento real, esta coluna viria depois da intervencao. Aqui ela e simulada para fins didaticos.

In [ ]:
base = 0.45
efeito_lembrete = 0.14
bonus_historico = 0.07 * dados["listas_anteriores_no_prazo"]
penalidade_carga = 0.006 * (dados["creditos_semestre"] - 20)

prob_entrega = (
    base
    + efeito_lembrete * (dados["grupo"] == "lembrete")
    + bonus_historico
    - penalidade_carga
).clip(0.05, 0.95)

dados["entregou_no_prazo"] = rng.binomial(1, prob_entrega)
dados[["estudante", "grupo", "entregou_no_prazo"]].head()

## 5. Comparando tratamento e controle

A primeira comparacao e a taxa de entrega no prazo em cada grupo.

In [ ]:
taxas = dados.groupby("grupo")["entregou_no_prazo"].mean().sort_index()
diferenca = taxas["lembrete"] - taxas["controle"]

pd.DataFrame({
    "taxa_entrega_no_prazo": taxas,
    "percentual": taxas * 100
}).round(1)

In [ ]:
ax = (taxas * 100).plot(
    kind="bar",
    color=["#4c78a8", "#f58518"],
    figsize=(6, 4),
    ylim=(0, 100),
    title="Entrega no prazo por grupo"
)
ax.set_ylabel("% de estudantes")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)

for i, valor in enumerate(taxas * 100):
    ax.text(i, valor + 2, f"{valor:.1f}%", ha="center")

plt.tight_layout()

## 6. A diferenca observada

A diferenca de taxas e uma forma simples de resumir o resultado. Ela mede quantos pontos percentuais separam o grupo tratado do grupo controle.

In [ ]:
print(f"Diferenca observada: {100 * diferenca:.1f} pontos percentuais")

## 7. Interpretacao causal

Como o grupo foi sorteado, a comparacao entre tratamento e controle e mais justa do que uma comparacao entre pessoas que escolheram receber ou nao receber lembrete.

Mesmo assim, a conclusao deve ser cuidadosa: neste exemplo, observamos uma diferenca positiva associada ao lembrete. Para afirmar um efeito em uma turma real, precisamos olhar o desenho do experimento, possiveis problemas de implementacao e se houve vazamento entre grupos.

## 8. O que poderia dar errado?

- Alguns estudantes do controle podem receber o lembrete por colegas.
- O Moodle pode nao entregar todas as mensagens.
- Entregar no prazo nao mede aprendizagem diretamente.
- O efeito pode ser diferente em outra turma ou outro tipo de lista.
- A intervencao pode ajudar pouco se o problema principal for dificuldade com o conteudo.

## Para praticar

1. Identifique tratamento, controle, resultado e unidade observacional.
2. Explique por que sortear estudantes ajuda na interpretacao causal.
3. Mude `efeito_lembrete` e observe como o grafico muda.
4. Escreva uma limitacao que nao aparece na lista acima.